In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv(
    "../data/processed/vijayawada_air_quality_clean.csv"
)

df["collected_at"] = pd.to_datetime(
    df["collected_at"]
)

df = df.sort_values(
    ["station_id", "collected_at"]
).reset_index(drop=True)

print("Shape:", df.shape)
display(df.head())

Shape: (7100, 7)


,station_id,collected_at,PM10,NO2,SO2,CO,PM2.5
0,site_5685,2024-01-01 00:00:00,64.82,13.18,2.5,0.78,39.16
1,site_5685,2024-01-01 01:00:00,55.31,13.93,2.5,0.78,34.52
2,site_5685,2024-01-01 02:00:00,35.92,13.74,2.5,0.78,25.06
3,site_5685,2024-01-01 03:00:00,29.16,12.48,2.5,0.78,21.77
4,site_5685,2024-01-01 04:00:00,28.25,12.06,2.5,0.78,21.32


In [2]:
print("Columns:")
print(df.columns.tolist())

print("\nStations:")
print(df["station_id"].unique())

print("\nDate range:")
print(df["collected_at"].min())
print(df["collected_at"].max())

print("\nMissing values:")
print(df.isnull().sum())

Columns:
['station_id', 'collected_at', 'PM10', 'NO2', 'SO2', 'CO', 'PM2.5']

Stations:
['site_5685' 'site_5848' 'site_5849' 'site_5981']

Date range:
2024-01-01 00:00:00
2024-03-31 23:00:00

Missing values:
station_id      0
collected_at    0
PM10            0
NO2             0
SO2             0
CO              0
PM2.5           0
dtype: int64


In [3]:
df["target_pm25"] = (
    df.groupby("station_id")["PM2.5"]
      .shift(-1)
)

In [4]:
display(
    df[
        [
            "station_id",
            "collected_at",
            "PM2.5",
            "target_pm25"
        ]
    ].head(10)
)

,station_id,collected_at,PM2.5,target_pm25
0,site_5685,2024-01-01 00:00:00,39.16,34.52
1,site_5685,2024-01-01 01:00:00,34.52,25.06
2,site_5685,2024-01-01 02:00:00,25.06,21.77
3,site_5685,2024-01-01 03:00:00,21.77,21.32
4,site_5685,2024-01-01 04:00:00,21.32,20.17
5,site_5685,2024-01-01 05:00:00,20.17,18.14
6,site_5685,2024-01-01 06:00:00,18.14,18.21
7,site_5685,2024-01-01 07:00:00,18.21,44.89
8,site_5685,2024-01-01 08:00:00,44.89,65.45
9,site_5685,2024-01-01 09:00:00,65.45,55.55


In [5]:
lag_hours = [1, 2, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"pm25_lag_{lag}h"] = (
        df.groupby("station_id")["PM2.5"]
          .shift(lag)
    )

In [6]:
display(
    df[
        [
            "collected_at",
            "PM2.5",
            "pm25_lag_1h",
            "pm25_lag_2h",
            "pm25_lag_3h",
            "pm25_lag_6h",
            "pm25_lag_12h",
            "pm25_lag_24h"
        ]
    ].head(30)
)

,collected_at,PM2.5,pm25_lag_1h,pm25_lag_2h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h
0,2024-01-01 00:00:00,39.16,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01 01:00:00,34.52,39.16,NaN,NaN,NaN,NaN,NaN
2,2024-01-01 02:00:00,25.06,34.52,39.16,NaN,NaN,NaN,NaN
3,2024-01-01 03:00:00,21.77,25.06,34.52,39.16,NaN,NaN,NaN
4,2024-01-01 04:00:00,21.32,21.77,25.06,34.52,NaN,NaN,NaN
5,2024-01-01 05:00:00,20.17,21.32,21.77,25.06,NaN,NaN,NaN
6,2024-01-01 06:00:00,18.14,20.17,21.32,21.77,39.16,NaN,NaN
7,2024-01-01 07:00:00,18.21,18.14,20.17,21.32,34.52,NaN,NaN
8,2024-01-01 08:00:00,44.89,18.21,18.14,20.17,25.06,NaN,NaN
9,2024-01-01 09:00:00,65.45,44.89,18.21,18.14,21.77,NaN,NaN


In [7]:
rolling_windows = [3, 6, 12, 24]

for window in rolling_windows:
    df[f"pm25_rolling_mean_{window}h"] = (
        df.groupby("station_id")["PM2.5"]
          .transform(
              lambda x: x.shift(1).rolling(window).mean()
          )
    )

In [8]:
df["hour"] = df["collected_at"].dt.hour
df["day_of_week"] = df["collected_at"].dt.dayofweek
df["month"] = df["collected_at"].dt.month
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [9]:
display(
    df[
        [
            "collected_at",
            "hour",
            "day_of_week",
            "month",
            "is_weekend"
        ]
    ].head()
)

,collected_at,hour,day_of_week,month,is_weekend
0,2024-01-01 00:00:00,0,0,1,0
1,2024-01-01 01:00:00,1,0,1,0
2,2024-01-01 02:00:00,2,0,1,0
3,2024-01-01 03:00:00,3,0,1,0
4,2024-01-01 04:00:00,4,0,1,0


In [10]:
pollutants = [
    "PM10",
    "NO2",
    "SO2",
    "CO"
]

for pollutant in pollutants:

    df[f"{pollutant}_lag_1h"] = (
        df.groupby("station_id")[pollutant]
          .shift(1)
    )

In [11]:
print("Shape after feature engineering:")
print(df.shape)

display(df.head())

Shape after feature engineering:
(7100, 26)


,station_id,collected_at,PM10,NO2,SO2,CO,PM2.5,target_pm25,pm25_lag_1h,pm25_lag_2h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h,pm25_rolling_mean_3h,pm25_rolling_mean_6h,pm25_rolling_mean_12h,pm25_rolling_mean_24h,hour,day_of_week,month,is_weekend,PM10_lag_1h,NO2_lag_1h,SO2_lag_1h,CO_lag_1h
0,site_5685,2024-01-01 00:00:00,64.82,13.18,2.5,0.78,39.16,34.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,NaN,NaN,NaN,NaN
1,site_5685,2024-01-01 01:00:00,55.31,13.93,2.5,0.78,34.52,25.06,39.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,0,64.82,13.18,2.5,0.78
2,site_5685,2024-01-01 02:00:00,35.92,13.74,2.5,0.78,25.06,21.77,34.52,39.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,1,0,55.31,13.93,2.5,0.78
3,site_5685,2024-01-01 03:00:00,29.16,12.48,2.5,0.78,21.77,21.32,25.06,34.52,39.16,NaN,NaN,NaN,32.913333,NaN,NaN,NaN,3,0,1,0,35.92,13.74,2.5,0.78
4,site_5685,2024-01-01 04:00:00,28.25,12.06,2.5,0.78,21.32,20.17,21.77,25.06,34.52,NaN,NaN,NaN,27.116667,NaN,NaN,NaN,4,0,1,0,29.16,12.48,2.5,0.78


In [12]:
print("Missing values created by lagging:")
display(
    df.isnull().sum()
    .sort_values(ascending=False)
    .head(20)
)

Missing values created by lagging:


pm25_lag_24h             96
pm25_rolling_mean_24h    96
pm25_lag_12h             48
pm25_rolling_mean_12h    48
pm25_rolling_mean_6h     24
pm25_lag_6h              24
pm25_rolling_mean_3h     12
pm25_lag_3h              12
pm25_lag_2h               8
SO2_lag_1h                4
NO2_lag_1h                4
PM10_lag_1h               4
CO_lag_1h                 4
pm25_lag_1h               4
target_pm25               4
collected_at              0
PM2.5                     0
CO                        0
hour                      0
day_of_week               0
dtype: int64

In [13]:
feature_columns = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "pm25_lag_1h",
    "pm25_lag_2h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    "pm25_rolling_mean_3h",
    "pm25_rolling_mean_6h",
    "pm25_rolling_mean_12h",
    "pm25_rolling_mean_24h",
    "PM10_lag_1h",
    "NO2_lag_1h",
    "SO2_lag_1h",
    "CO_lag_1h",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]

target_column = "target_pm25"

model_data = df.dropna(
    subset=feature_columns + [target_column]
).copy()

print("Original rows:", len(df))
print("Modeling rows:", len(model_data))
print("Rows removed:", len(df) - len(model_data))

Original rows: 7100
Modeling rows: 7000
Rows removed: 100


In [14]:
model_data.to_csv(
    "../data/processed/vijayawada_features.csv",
    index=False
)

print("Feature dataset saved successfully.")

Feature dataset saved successfully.


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [26]:
# Reload clean multivariate data
df = pd.read_csv(
    "../data/processed/vijayawada_air_quality_clean.csv"
)

df["collected_at"] = pd.to_datetime(df["collected_at"])

df = df.sort_values(
    ["station_id", "collected_at"]
).reset_index(drop=True)

# Create a continuous hourly grid for each station
station_frames = []

for station, group in df.groupby("station_id"):

    group = group.set_index("collected_at")

    hourly_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="h"
    )

    group = group.reindex(hourly_index)

    group["station_id"] = station
    group.index.name = "collected_at"

    station_frames.append(group.reset_index())

df_hourly = pd.concat(
    station_frames,
    ignore_index=True
)

print("Hourly dataset shape:", df_hourly.shape)
display(df_hourly.head())

Hourly dataset shape: (7955, 7)


,collected_at,station_id,PM10,NO2,SO2,CO,PM2.5
0,2024-01-01 00:00:00,site_5685,64.82,13.18,2.5,0.78,39.16
1,2024-01-01 01:00:00,site_5685,55.31,13.93,2.5,0.78,34.52
2,2024-01-01 02:00:00,site_5685,35.92,13.74,2.5,0.78,25.06
3,2024-01-01 03:00:00,site_5685,29.16,12.48,2.5,0.78,21.77
4,2024-01-01 04:00:00,site_5685,28.25,12.06,2.5,0.78,21.32


In [27]:
df_hourly["target_pm25"] = (
    df_hourly.groupby("station_id")["PM2.5"]
    .shift(-1)
)

In [28]:
df_hourly["next_time"] = (
    df_hourly.groupby("station_id")["collected_at"]
    .shift(-1)
)

df_hourly["valid_target"] = (
    df_hourly["next_time"]
    == df_hourly["collected_at"] + pd.Timedelta(hours=1)
)

In [29]:
df_hourly.loc[
    ~df_hourly["valid_target"],
    "target_pm25"
] = np.nan

In [30]:
lag_hours = [1, 2, 3, 6, 12, 24]

for lag in lag_hours:

    df_hourly[f"pm25_lag_{lag}h"] = (
        df_hourly.groupby("station_id")["PM2.5"]
        .shift(lag)
    )

    expected_time = (
        df_hourly["collected_at"]
        - pd.to_timedelta(lag, unit="h")
    )

    actual_time = (
        df_hourly.groupby("station_id")["collected_at"]
        .shift(lag)
    )

    invalid = actual_time != expected_time

    df_hourly.loc[
        invalid,
        f"pm25_lag_{lag}h"
    ] = np.nan

In [31]:
rolling_windows = [3, 6, 12, 24]

for window in rolling_windows:

    df_hourly[f"pm25_rolling_mean_{window}h"] = (
        df_hourly.groupby("station_id")["PM2.5"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window)
             .mean()
        )
    )

In [32]:
df_hourly["hour"] = df_hourly["collected_at"].dt.hour
df_hourly["day_of_week"] = df_hourly["collected_at"].dt.dayofweek
df_hourly["month"] = df_hourly["collected_at"].dt.month
df_hourly["is_weekend"] = (
    df_hourly["day_of_week"] >= 5
).astype(int)

In [33]:
pollutants = ["PM10", "NO2", "SO2", "CO"]

for pollutant in pollutants:

    df_hourly[f"{pollutant}_lag_1h"] = (
        df_hourly.groupby("station_id")[pollutant]
        .shift(1)
    )

In [34]:
feature_columns = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "pm25_lag_1h",
    "pm25_lag_2h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    "pm25_rolling_mean_3h",
    "pm25_rolling_mean_6h",
    "pm25_rolling_mean_12h",
    "pm25_rolling_mean_24h",
    "PM10_lag_1h",
    "NO2_lag_1h",
    "SO2_lag_1h",
    "CO_lag_1h",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]

model_data = df_hourly.dropna(
    subset=feature_columns + ["target_pm25"]
).copy()

print("Original hourly rows:", len(df_hourly))
print("Modeling rows:", len(model_data))
print("Rows removed:", len(df_hourly) - len(model_data))

Original hourly rows: 7955
Modeling rows: 5800
Rows removed: 2155


In [35]:
# Save the final gap-aware feature dataset

output_path = "../data/processed/vijayawada_features.csv"

model_data.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final modeling dataset shape:", model_data.shape)

Saved: ../data/processed/vijayawada_features.csv
Final modeling dataset shape: (5800, 28)


In [36]:
print(model_data.head())
print("\nColumns:")
print(model_data.columns.tolist())

          collected_at station_id   PM10    NO2   SO2    CO  PM2.5  \
24 2024-01-02 00:00:00  site_5685  27.19  15.30  1.23  0.39  20.80   
25 2024-01-02 01:00:00  site_5685  23.76  15.32  1.23  0.39  19.13   
26 2024-01-02 02:00:00  site_5685  22.66  14.42  1.23  0.39  18.60   
27 2024-01-02 03:00:00  site_5685  24.84  14.39  1.23  0.39  19.66   
28 2024-01-02 04:00:00  site_5685  24.71  14.23  1.23  0.39  19.60   

    target_pm25           next_time  valid_target  pm25_lag_1h  pm25_lag_2h  \
24        19.13 2024-01-02 01:00:00          True        28.94        28.06   
25        18.60 2024-01-02 02:00:00          True        20.80        28.94   
26        19.66 2024-01-02 03:00:00          True        19.13        20.80   
27        19.60 2024-01-02 04:00:00          True        18.60        19.13   
28        18.09 2024-01-02 05:00:00          True        19.66        18.60   

    pm25_lag_3h  pm25_lag_6h  pm25_lag_12h  pm25_lag_24h  \
24        22.04        57.09         45.95  

In [37]:
# Make sure data is sorted chronologically
model_data = model_data.sort_values("collected_at").reset_index(drop=True)

# Define dates
train_end = "2024-02-01"
val_end = "2024-03-01"

train_data = model_data[
    model_data["collected_at"] < train_end
]

val_data = model_data[
    (model_data["collected_at"] >= train_end) &
    (model_data["collected_at"] < val_end)
]

test_data = model_data[
    model_data["collected_at"] >= val_end
]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (1513, 28)
Validation: (1890, 28)
Test: (2397, 28)


In [38]:
print("TRAIN")
print(train_data["collected_at"].min())
print(train_data["collected_at"].max())

print("\nVALIDATION")
print(val_data["collected_at"].min())
print(val_data["collected_at"].max())

print("\nTEST")
print(test_data["collected_at"].min())
print(test_data["collected_at"].max())

TRAIN
2024-01-02 00:00:00
2024-01-31 23:00:00

VALIDATION
2024-02-01 00:00:00
2024-02-29 23:00:00

TEST
2024-03-01 00:00:00
2024-03-31 22:00:00


In [39]:
feature_columns = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "pm25_lag_1h",
    "pm25_lag_2h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    "pm25_rolling_mean_3h",
    "pm25_rolling_mean_6h",
    "pm25_rolling_mean_12h",
    "pm25_rolling_mean_24h",
    "PM10_lag_1h",
    "NO2_lag_1h",
    "SO2_lag_1h",
    "CO_lag_1h",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]

target_column = "target_pm25"

X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_val = val_data[feature_columns]
y_val = val_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1513, 23)
y_train: (1513,)
X_val: (1890, 23)
y_val: (1890,)
X_test: (2397, 23)
y_test: (2397,)


In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Persistence prediction
y_pred_baseline = X_test["PM2.5"]

# Metrics
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_baseline)
)

baseline_r2 = r2_score(y_test, y_pred_baseline)

print("Persistence Baseline")
print("--------------------")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R²   : {baseline_r2:.4f}")

Persistence Baseline
--------------------
MAE  : 2.4732
RMSE : 4.1057
R²   : 0.6500


In [41]:
baseline_results = pd.DataFrame({
    "Model": ["Persistence Baseline"],
    "MAE": [baseline_mae],
    "RMSE": [baseline_rmse],
    "R2": [baseline_r2]
})

baseline_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,2.473187,4.105665,0.650018


In [42]:
from sklearn.linear_model import LinearRegression

# Create model
linear_model = LinearRegression()

# Train only on training data
linear_model.fit(X_train, y_train)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


In [43]:
y_pred_val_lr = linear_model.predict(X_val)
y_pred_test_lr = linear_model.predict(X_test)

print("Validation predictions:", len(y_pred_val_lr))
print("Test predictions:", len(y_pred_test_lr))

Validation predictions: 1890
Test predictions: 2397


In [44]:
# Validation metrics
lr_val_mae = mean_absolute_error(y_val, y_pred_val_lr)
lr_val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val_lr))
lr_val_r2 = r2_score(y_val, y_pred_val_lr)

# Test metrics
lr_test_mae = mean_absolute_error(y_test, y_pred_test_lr)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test_lr))
lr_test_r2 = r2_score(y_test, y_pred_test_lr)

print("Linear Regression - Validation")
print("--------------------------------")
print(f"MAE  : {lr_val_mae:.4f}")
print(f"RMSE : {lr_val_rmse:.4f}")
print(f"R²   : {lr_val_r2:.4f}")

print("\nLinear Regression - Test")
print("-------------------------")
print(f"MAE  : {lr_test_mae:.4f}")
print(f"RMSE : {lr_test_rmse:.4f}")
print(f"R²   : {lr_test_r2:.4f}")

Linear Regression - Validation
--------------------------------
MAE  : 9.0452
RMSE : 24.8216
R²   : 0.1818

Linear Regression - Test
-------------------------
MAE  : 3.2045
RMSE : 4.6043
R²   : 0.5599


In [45]:
results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression"
    ],
    "MAE": [
        baseline_mae,
        lr_test_mae
    ],
    "RMSE": [
        baseline_rmse,
        lr_test_rmse
    ],
    "R2": [
        baseline_r2,
        lr_test_r2
    ]
})

results

,Model,MAE,RMSE,R2
0,Persistence Baseline,2.473187,4.105665,0.650018
1,Linear Regression,3.204504,4.604252,0.559854


In [46]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [47]:
y_pred_val_rf = rf_model.predict(X_val)
y_pred_test_rf = rf_model.predict(X_test)

print("Validation predictions:", len(y_pred_val_rf))
print("Test predictions:", len(y_pred_test_rf))

Validation predictions: 1890
Test predictions: 2397


In [48]:
rf_val_mae = mean_absolute_error(y_val, y_pred_val_rf)
rf_val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val_rf))
rf_val_r2 = r2_score(y_val, y_pred_val_rf)

rf_test_mae = mean_absolute_error(y_test, y_pred_test_rf)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))
rf_test_r2 = r2_score(y_test, y_pred_test_rf)

print("Random Forest - Validation")
print("---------------------------")
print(f"MAE  : {rf_val_mae:.4f}")
print(f"RMSE : {rf_val_rmse:.4f}")
print(f"R²   : {rf_val_r2:.4f}")

print("\nRandom Forest - Test")
print("---------------------")
print(f"MAE  : {rf_test_mae:.4f}")
print(f"RMSE : {rf_test_rmse:.4f}")
print(f"R²   : {rf_test_r2:.4f}")

Random Forest - Validation
---------------------------
MAE  : 10.6297
RMSE : 30.1872
R²   : -0.2101

Random Forest - Test
---------------------
MAE  : 2.8048
RMSE : 5.7379
R²   : 0.3164


In [49]:
results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        lr_test_mae,
        rf_test_mae
    ],
    "RMSE": [
        baseline_rmse,
        lr_test_rmse,
        rf_test_rmse
    ],
    "R2": [
        baseline_r2,
        lr_test_r2,
        rf_test_r2
    ]
})

results

,Model,MAE,RMSE,R2
0,Persistence Baseline,2.473187,4.105665,0.650018
1,Linear Regression,3.204504,4.604252,0.559854
2,Random Forest,2.804774,5.737864,0.316437


In [50]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

feature_importance

,Feature,Importance
0,PM2.5,0.454488
1,pm25_lag_24h,0.233849
2,pm25_lag_1h,0.039049
3,pm25_rolling_mean_24h,0.034295
4,pm25_rolling_mean_6h,0.032165
5,PM10_lag_1h,0.029614
6,pm25_rolling_mean_3h,0.027452
7,pm25_rolling_mean_12h,0.025293
8,PM10,0.018998
9,pm25_lag_2h,0.018009


In [51]:
feature_importance.head(10)

,Feature,Importance
0,PM2.5,0.454488
1,pm25_lag_24h,0.233849
2,pm25_lag_1h,0.039049
3,pm25_rolling_mean_24h,0.034295
4,pm25_rolling_mean_6h,0.032165
5,PM10_lag_1h,0.029614
6,pm25_rolling_mean_3h,0.027452
7,pm25_rolling_mean_12h,0.025293
8,PM10,0.018998
9,pm25_lag_2h,0.018009


In [52]:
error_analysis = test_data[
    ["collected_at", "station_id", "PM2.5", "target_pm25"]
].copy()

error_analysis["prediction"] = y_pred_test_rf

error_analysis["absolute_error"] = (
    abs(error_analysis["target_pm25"] - error_analysis["prediction"])
)

error_analysis = error_analysis.sort_values(
    "absolute_error",
    ascending=False
)

error_analysis.head(10)

,collected_at,station_id,PM2.5,target_pm25,prediction,absolute_error
3437,2024-03-01 08:00:00,site_5981,37.47,38.72,208.71540,169.99540
3911,2024-03-06 08:00:00,site_5848,51.12,62.18,164.03125,101.85125
3867,2024-03-05 20:00:00,site_5981,40.49,73.77,39.64245,34.12755
4391,2024-03-11 22:00:00,site_5981,82.85,35.47,67.62645,32.15645
3962,2024-03-07 01:00:00,site_5981,35.47,67.57,36.10850,31.46150
4384,2024-03-11 20:00:00,site_5981,43.95,73.24,42.08030,31.15970
3827,2024-03-05 10:00:00,site_5848,39.64,36.50,64.31555,27.81555
3812,2024-03-05 06:00:00,site_5848,45.97,71.78,45.30965,26.47035
3818,2024-03-05 07:00:00,site_5848,71.78,90.98,66.20440,24.77560
3820,2024-03-05 08:00:00,site_5848,90.98,42.02,66.41945,24.39945


In [53]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 6.5 MB/s eta 0:00:08
   - -------------------------------------- 2.4/48.9 MB 7.3 MB/s eta 0:00:07
   --- ------------------------------------ 4.2/48.9 MB 7.6 MB/s eta 0:00:06
   ---- ----------------------------------- 5.8/48.9 MB 7.7 MB/s eta 0:00:06
   ----- ---------------------------------- 6.6/48.9 MB 6.9 MB/s eta 0:00:07
   ------ --------------------------------- 7.6/48.9 MB 6.5 MB/s eta 0:00:07
   ------- -------------------------------- 8.7/48.9 MB 6.2 MB/s eta 0:00:07
   ------- -------------------------------- 9.4/48.9 MB 5.9 MB/s eta 0:00:07
   -------- ------------------------------- 10.5/48.9 MB 5.8 MB/s eta 0:00:07
   --------- ------------------------------ 11.5/48.9 MB 5.7 MB/s eta 0:00:07
   ---------- ----------------------------- 12.8/48.9 MB 5.7 MB/s eta 0:00:07
   ----------- ---------------------------- 13.9/48.9 MB 5.7 MB/s eta 0:00:07
  

In [54]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [55]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [56]:
y_pred_val_xgb = xgb_model.predict(X_val)
y_pred_test_xgb = xgb_model.predict(X_test)

print("Validation predictions:", len(y_pred_val_xgb))
print("Test predictions:", len(y_pred_test_xgb))

Validation predictions: 1890
Test predictions: 2397


In [57]:
xgb_val_mae = mean_absolute_error(y_val, y_pred_val_xgb)
xgb_val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val_xgb))
xgb_val_r2 = r2_score(y_val, y_pred_val_xgb)

xgb_test_mae = mean_absolute_error(y_test, y_pred_test_xgb)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test_xgb))
xgb_test_r2 = r2_score(y_test, y_pred_test_xgb)

print("XGBoost - Validation")
print("---------------------")
print(f"MAE  : {xgb_val_mae:.4f}")
print(f"RMSE : {xgb_val_rmse:.4f}")
print(f"R²   : {xgb_val_r2:.4f}")

print("\nXGBoost - Test")
print("----------------")
print(f"MAE  : {xgb_test_mae:.4f}")
print(f"RMSE : {xgb_test_rmse:.4f}")
print(f"R²   : {xgb_test_r2:.4f}")

XGBoost - Validation
---------------------
MAE  : 9.6639
RMSE : 27.9605
R²   : -0.0382

XGBoost - Test
----------------
MAE  : 2.6720
RMSE : 3.9295
R²   : 0.6794


In [58]:
results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        baseline_mae,
        lr_test_mae,
        rf_test_mae,
        xgb_test_mae
    ],
    "RMSE": [
        baseline_rmse,
        lr_test_rmse,
        rf_test_rmse,
        xgb_test_rmse
    ],
    "R2": [
        baseline_r2,
        lr_test_r2,
        rf_test_r2,
        xgb_test_r2
    ]
})

results

,Model,MAE,RMSE,R2
0,Persistence Baseline,2.473187,4.105665,0.650018
1,Linear Regression,3.204504,4.604252,0.559854
2,Random Forest,2.804774,5.737864,0.316437
3,XGBoost,2.672035,3.929542,0.679401


In [59]:
xgb_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": xgb_model.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

xgb_importance.head(15)

,Feature,Importance
0,day_of_week,0.314709
1,pm25_lag_24h,0.158204
2,pm25_lag_2h,0.130333
3,PM2.5,0.099848
4,pm25_rolling_mean_3h,0.073683
5,pm25_lag_1h,0.043456
6,pm25_lag_3h,0.028730
7,pm25_rolling_mean_6h,0.026104
8,pm25_rolling_mean_24h,0.022967
9,CO,0.017300


In [60]:
xgb_importance.head(10)

,Feature,Importance
0,day_of_week,0.314709
1,pm25_lag_24h,0.158204
2,pm25_lag_2h,0.130333
3,PM2.5,0.099848
4,pm25_rolling_mean_3h,0.073683
5,pm25_lag_1h,0.043456
6,pm25_lag_3h,0.028730
7,pm25_rolling_mean_6h,0.026104
8,pm25_rolling_mean_24h,0.022967
9,CO,0.017300


In [61]:
xgb_error_analysis = test_data[
    ["collected_at", "station_id", "PM2.5", "target_pm25"]
].copy()

xgb_error_analysis["prediction"] = y_pred_test_xgb

xgb_error_analysis["absolute_error"] = abs(
    xgb_error_analysis["target_pm25"] -
    xgb_error_analysis["prediction"]
)

xgb_error_analysis = xgb_error_analysis.sort_values(
    "absolute_error",
    ascending=False
)

xgb_error_analysis.head(10)

,collected_at,station_id,PM2.5,target_pm25,prediction,absolute_error
4384,2024-03-11 20:00:00,site_5981,43.95,73.24,39.620773,33.619227
3867,2024-03-05 20:00:00,site_5981,40.49,73.77,40.458809,33.311191
3962,2024-03-07 01:00:00,site_5981,35.47,67.57,35.780640,31.789360
3521,2024-03-02 05:00:00,site_5981,40.14,63.92,38.054245,25.865755
3820,2024-03-05 08:00:00,site_5848,90.98,42.02,67.248108,25.228108
3818,2024-03-05 07:00:00,site_5848,71.78,90.98,66.678062,24.301938
3812,2024-03-05 06:00:00,site_5848,45.97,71.78,47.934204,23.845796
4391,2024-03-11 22:00:00,site_5981,82.85,35.47,57.970409,22.500409
4388,2024-03-11 21:00:00,site_5981,73.24,82.85,60.483616,22.366384
5194,2024-03-21 19:00:00,site_5981,27.56,50.86,28.753731,22.106269


In [62]:
xgb_configs = [
    {
        "name": "XGB_1",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "XGB_2",
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "XGB_3",
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "XGB_4",
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
]

print("Configurations:", len(xgb_configs))

Configurations: 4


In [63]:
tuning_results = []

for config in xgb_configs:

    model = XGBRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, val_predictions)
    rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
    r2 = r2_score(y_val, val_predictions)

    tuning_results.append({
        "Model": config["name"],
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

tuning_results = pd.DataFrame(tuning_results)

tuning_results.sort_values("MAE")

,Model,MAE,RMSE,R2
3,XGB_4,9.528078,27.803068,-0.026541
1,XGB_2,9.584454,27.406935,0.002503
2,XGB_3,9.645923,27.967752,-0.038738
0,XGB_1,9.650066,28.422278,-0.072775


In [64]:
print("TRAIN TARGET")
print(y_train.describe())

print("\nVALIDATION TARGET")
print(y_val.describe())

print("\nTEST TARGET")
print(y_test.describe())

TRAIN TARGET
count    1513.000000
mean       39.522056
std        22.416615
min        14.210000
25%        30.190000
50%        34.980000
75%        45.160000
max       508.210000
Name: target_pm25, dtype: float64

VALIDATION TARGET
count    1890.000000
mean       42.901275
std        27.448560
min         0.100000
25%        32.807500
50%        37.445000
75%        45.097500
max       549.260000
Name: target_pm25, dtype: float64

TEST TARGET
count    2397.000000
mean       37.470734
std         6.941471
min        16.980000
25%        33.340000
50%        37.070000
75%        41.640000
max        90.980000
Name: target_pm25, dtype: float64


In [65]:
print("Validation target by station:")
print(
    val_data.groupby("station_id")["target_pm25"]
    .agg(["count", "mean", "std", "min", "max"])
)

print("\nTest target by station:")
print(
    test_data.groupby("station_id")["target_pm25"]
    .agg(["count", "mean", "std", "min", "max"])
)

Validation target by station:
            count       mean        std    min     max
station_id                                            
site_5685     270  40.240481   5.306495  30.38   54.13
site_5848     585  35.890598  13.817790  15.23  156.92
site_5849     634  37.986625   7.483643  16.16   63.99
site_5981     401  62.690673  51.615645   0.10  549.26

Test target by station:
            count       mean       std    min    max
station_id                                          
site_5685     463  38.883844  5.323166  27.36  60.79
site_5848     743  40.353917  6.737594  25.94  90.98
site_5849     714  33.747563  6.667955  16.98  56.25
site_5981     477  37.181153  6.471475  25.47  82.85


In [66]:
val_persistence_pred = X_val["PM2.5"]

val_persistence_mae = mean_absolute_error(
    y_val,
    val_persistence_pred
)

val_persistence_rmse = np.sqrt(
    mean_squared_error(y_val, val_persistence_pred)
)

val_persistence_r2 = r2_score(
    y_val,
    val_persistence_pred
)

print("Persistence - Validation")
print("-------------------------")
print(f"MAE  : {val_persistence_mae:.4f}")
print(f"RMSE : {val_persistence_rmse:.4f}")
print(f"R²   : {val_persistence_r2:.4f}")

Persistence - Validation
-------------------------
MAE  : 8.4503
RMSE : 26.0537
R²   : 0.0986


In [67]:
extreme_validation = val_data[
    val_data["target_pm25"] > 150
].copy()

extreme_validation[
    [
        "collected_at",
        "station_id",
        "PM2.5",
        "target_pm25",
        "pm25_lag_1h",
        "pm25_lag_2h",
        "pm25_lag_24h"
    ]
].sort_values("target_pm25", ascending=False)

,collected_at,station_id,PM2.5,target_pm25,pm25_lag_1h,pm25_lag_2h,pm25_lag_24h
2785,2024-02-22 03:00:00,site_5981,176.14,549.26,100.03,101.02,99.08
3338,2024-02-29 07:00:00,site_5981,36.31,427.12,76.74,43.45,12.92
3256,2024-02-28 11:00:00,site_5981,36.66,352.31,39.08,49.58,63.48
3120,2024-02-27 01:00:00,site_5981,314.80,346.16,150.34,131.52,12.00
3118,2024-02-27 00:00:00,site_5981,150.34,314.80,131.52,59.56,27.70
2725,2024-02-21 07:00:00,site_5981,102.74,277.40,83.32,55.54,68.80
1920,2024-02-07 18:00:00,site_5981,87.76,227.58,18.58,11.82,92.42
3228,2024-02-28 04:00:00,site_5981,126.02,196.08,14.93,27.47,51.00
2781,2024-02-22 02:00:00,site_5981,100.03,176.14,101.02,108.14,93.09
2532,2024-02-18 03:00:00,site_5981,174.50,175.99,59.33,58.45,31.75


In [68]:
extreme_validation[
    [
        "collected_at",
        "station_id",
        "PM2.5",
        "PM10",
        "NO2",
        "SO2",
        "CO",
        "target_pm25"
    ]
].sort_values("target_pm25", ascending=False)

,collected_at,station_id,PM2.5,PM10,NO2,SO2,CO,target_pm25
2785,2024-02-22 03:00:00,site_5981,176.14,293.57,7.65,6.45,0.47,549.26
3338,2024-02-29 07:00:00,site_5981,36.31,62.52,5.44,13.33,0.80,427.12
3256,2024-02-28 11:00:00,site_5981,36.66,69.31,5.90,25.87,0.56,352.31
3120,2024-02-27 01:00:00,site_5981,314.80,513.67,6.88,11.68,0.50,346.16
3118,2024-02-27 00:00:00,site_5981,150.34,239.57,8.00,12.86,0.50,314.80
2725,2024-02-21 07:00:00,site_5981,102.74,171.24,4.96,11.33,1.04,277.40
1920,2024-02-07 18:00:00,site_5981,87.76,151.21,14.71,7.42,0.90,227.58
3228,2024-02-28 04:00:00,site_5981,126.02,210.03,7.51,10.20,0.50,196.08
2781,2024-02-22 02:00:00,site_5981,100.03,166.71,8.46,8.98,0.52,176.14
2532,2024-02-18 03:00:00,site_5981,174.50,349.01,7.92,8.76,0.52,175.99


In [69]:
extreme_validation[
    [
        "collected_at",
        "station_id",
        "PM2.5",
        "PM10",
        "NO2",
        "SO2",
        "CO",
        "target_pm25"
    ]
].sort_values("target_pm25", ascending=False)

,collected_at,station_id,PM2.5,PM10,NO2,SO2,CO,target_pm25
2785,2024-02-22 03:00:00,site_5981,176.14,293.57,7.65,6.45,0.47,549.26
3338,2024-02-29 07:00:00,site_5981,36.31,62.52,5.44,13.33,0.80,427.12
3256,2024-02-28 11:00:00,site_5981,36.66,69.31,5.90,25.87,0.56,352.31
3120,2024-02-27 01:00:00,site_5981,314.80,513.67,6.88,11.68,0.50,346.16
3118,2024-02-27 00:00:00,site_5981,150.34,239.57,8.00,12.86,0.50,314.80
2725,2024-02-21 07:00:00,site_5981,102.74,171.24,4.96,11.33,1.04,277.40
1920,2024-02-07 18:00:00,site_5981,87.76,151.21,14.71,7.42,0.90,227.58
3228,2024-02-28 04:00:00,site_5981,126.02,210.03,7.51,10.20,0.50,196.08
2781,2024-02-22 02:00:00,site_5981,100.03,166.71,8.46,8.98,0.52,176.14
2532,2024-02-18 03:00:00,site_5981,174.50,349.01,7.92,8.76,0.52,175.99


In [70]:
print("Targets > 100:", (y_val > 100).sum())
print("Targets > 150:", (y_val > 150).sum())
print("Targets > 200:", (y_val > 200).sum())

Targets > 100: 46
Targets > 150: 17
Targets > 200: 7


In [71]:
print("Extreme targets by station:")
print(
    val_data[val_data["target_pm25"] > 100]
    .groupby("station_id")
    .size()
)

Extreme targets by station:
station_id
site_5848     8
site_5981    38
dtype: int64


In [72]:
print("\nExtreme targets by date:")
print(
    val_data[val_data["target_pm25"] > 100]
    .assign(date=lambda x: x["collected_at"].dt.date)
    .groupby("date")
    .size()
)


Extreme targets by date:
date
2024-02-05    2
2024-02-06    3
2024-02-07    6
2024-02-10    1
2024-02-11    5
2024-02-12    1
2024-02-15    1
2024-02-16    2
2024-02-17    3
2024-02-18    3
2024-02-19    2
2024-02-20    1
2024-02-21    4
2024-02-22    4
2024-02-26    2
2024-02-27    2
2024-02-28    3
2024-02-29    1
dtype: int64


In [73]:
print("\nExtreme target statistics:")
print(
    val_data[val_data["target_pm25"] > 100]["target_pm25"]
    .describe()
)


Extreme target statistics:
count     46.000000
mean     164.095652
std       94.422786
min      100.030000
25%      105.267500
50%      128.175000
75%      171.147500
max      549.260000
Name: target_pm25, dtype: float64


In [74]:
normal_mask = y_val <= 100

normal_val_mae = mean_absolute_error(
    y_val[normal_mask],
    X_val.loc[normal_mask, "PM2.5"]
)

normal_val_rmse = np.sqrt(
    mean_squared_error(
        y_val[normal_mask],
        X_val.loc[normal_mask, "PM2.5"]
    )
)

print("Persistence Validation — targets <= 100")
print("-----------------------------------------")
print(f"MAE  : {normal_val_mae:.4f}")
print(f"RMSE : {normal_val_rmse:.4f}")
print(f"Rows : {normal_mask.sum()}")

Persistence Validation — targets <= 100
-----------------------------------------
MAE  : 6.6996
RMSE : 18.9766
Rows : 1844


In [75]:
# Create station one-hot encoded columns

station_dummies = pd.get_dummies(
    model_data["station_id"],
    prefix="station",
    dtype=int
)

station_dummies.head()

,station_site_5685,station_site_5848,station_site_5849,station_site_5981
0,1,0,0,0
1,0,0,1,0
2,1,0,0,0
3,0,0,1,0
4,1,0,0,0


In [76]:
station_features = station_dummies.columns.tolist()

feature_columns_station = feature_columns + station_features

print("Original features:", len(feature_columns))
print("Station features:", len(station_features))
print("Total features:", len(feature_columns_station))

Original features: 23
Station features: 4
Total features: 27


In [77]:
model_data_station = pd.concat(
    [
        model_data.reset_index(drop=True),
        station_dummies.reset_index(drop=True)
    ],
    axis=1
)

print(model_data_station.shape)

(5800, 32)


In [78]:
model_data_station = model_data_station.sort_values(
    "collected_at"
).reset_index(drop=True)

train_data_s = model_data_station[
    model_data_station["collected_at"] < "2024-02-01"
]

val_data_s = model_data_station[
    (model_data_station["collected_at"] >= "2024-02-01") &
    (model_data_station["collected_at"] < "2024-03-01")
]

test_data_s = model_data_station[
    model_data_station["collected_at"] >= "2024-03-01"
]

X_train_s = train_data_s[feature_columns_station]
y_train_s = train_data_s[target_column]

X_val_s = val_data_s[feature_columns_station]
y_val_s = val_data_s[target_column]

X_test_s = test_data_s[feature_columns_station]
y_test_s = test_data_s[target_column]

print("Train:", X_train_s.shape)
print("Validation:", X_val_s.shape)
print("Test:", X_test_s.shape)

Train: (1513, 27)
Validation: (1890, 27)
Test: (2397, 27)


In [79]:
print(X_train_s.dtypes.tail(4))

station_site_5685    int64
station_site_5848    int64
station_site_5849    int64
station_site_5981    int64
dtype: object


In [80]:
from xgboost import XGBRegressor

xgb_station = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_station.fit(
    X_train_s,
    y_train_s
)

print("XGBoost with station features trained successfully.")

XGBoost with station features trained successfully.


In [82]:
y_pred_val_xgb_s = xgb_station.predict(X_val_s)
y_pred_test_xgb_s = xgb_station.predict(X_test_s)

In [83]:
xgb_s_val_mae = mean_absolute_error(
    y_val_s,
    y_pred_val_xgb_s
)

xgb_s_val_rmse = np.sqrt(
    mean_squared_error(y_val_s, y_pred_val_xgb_s)
)

xgb_s_val_r2 = r2_score(
    y_val_s,
    y_pred_val_xgb_s
)

xgb_s_test_mae = mean_absolute_error(
    y_test_s,
    y_pred_test_xgb_s
)

xgb_s_test_rmse = np.sqrt(
    mean_squared_error(y_test_s, y_pred_test_xgb_s)
)

xgb_s_test_r2 = r2_score(
    y_test_s,
    y_pred_test_xgb_s
)

print("XGBoost + Station - Validation")
print("--------------------------------")
print(f"MAE  : {xgb_s_val_mae:.4f}")
print(f"RMSE : {xgb_s_val_rmse:.4f}")
print(f"R²   : {xgb_s_val_r2:.4f}")

print("\nXGBoost + Station - Test")
print("-------------------------")
print(f"MAE  : {xgb_s_test_mae:.4f}")
print(f"RMSE : {xgb_s_test_rmse:.4f}")
print(f"R²   : {xgb_s_test_r2:.4f}")

XGBoost + Station - Validation
--------------------------------
MAE  : 9.7690
RMSE : 28.5323
R²   : -0.0811

XGBoost + Station - Test
-------------------------
MAE  : 2.8084
RMSE : 4.0738
R²   : 0.6554


In [84]:
station_comparison = pd.DataFrame({
    "Model": [
        "Persistence",
        "XGBoost",
        "XGBoost + Station"
    ],
    "MAE": [
        baseline_mae,
        xgb_test_mae,
        xgb_s_test_mae
    ],
    "RMSE": [
        baseline_rmse,
        xgb_test_rmse,
        xgb_s_test_rmse
    ],
    "R2": [
        baseline_r2,
        xgb_test_r2,
        xgb_s_test_r2
    ]
})

station_comparison

,Model,MAE,RMSE,R2
0,Persistence,2.473187,4.105665,0.650018
1,XGBoost,2.672035,3.929542,0.679401
2,XGBoost + Station,2.808350,4.073778,0.655434


In [85]:
final_error = test_data[
    ["collected_at", "station_id", "PM2.5", "target_pm25"]
].copy()

final_error["prediction"] = y_pred_test_xgb

final_error["absolute_error"] = abs(
    final_error["target_pm25"] -
    final_error["prediction"]
)

print("Total test rows:", len(final_error))

print("\nMAE:")
print(final_error["absolute_error"].mean())

print("\nErrors > 5:")
print((final_error["absolute_error"] > 5).sum())

print("\nErrors > 10:")
print((final_error["absolute_error"] > 10).sum())

print("\nErrors > 20:")
print((final_error["absolute_error"] > 20).sum())

Total test rows: 2397

MAE:
2.672035077269693

Errors > 5:
309

Errors > 10:
57

Errors > 20:
11


In [86]:
print("\nLargest prediction errors:")
print(
    final_error
    .sort_values("absolute_error", ascending=False)
    .head(10)
)


Largest prediction errors:
            collected_at station_id  PM2.5  target_pm25  prediction  \
4384 2024-03-11 20:00:00  site_5981  43.95        73.24   39.620773   
3867 2024-03-05 20:00:00  site_5981  40.49        73.77   40.458809   
3962 2024-03-07 01:00:00  site_5981  35.47        67.57   35.780640   
3521 2024-03-02 05:00:00  site_5981  40.14        63.92   38.054245   
3820 2024-03-05 08:00:00  site_5848  90.98        42.02   67.248108   
3818 2024-03-05 07:00:00  site_5848  71.78        90.98   66.678062   
3812 2024-03-05 06:00:00  site_5848  45.97        71.78   47.934204   
4391 2024-03-11 22:00:00  site_5981  82.85        35.47   57.970409   
4388 2024-03-11 21:00:00  site_5981  73.24        82.85   60.483616   
5194 2024-03-21 19:00:00  site_5981  27.56        50.86   28.753731   

      absolute_error  
4384       33.619227  
3867       33.311191  
3962       31.789360  
3521       25.865755  
3820       25.228108  
3818       24.301938  
3812       23.845796  
4391  

In [87]:
final_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": xgb_model.feature_importances_
})

final_importance = final_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

final_importance

,Feature,Importance
0,day_of_week,0.314709
1,pm25_lag_24h,0.158204
2,pm25_lag_2h,0.130333
3,PM2.5,0.099848
4,pm25_rolling_mean_3h,0.073683
5,pm25_lag_1h,0.043456
6,pm25_lag_3h,0.028730
7,pm25_rolling_mean_6h,0.026104
8,pm25_rolling_mean_24h,0.022967
9,CO,0.017300


In [88]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

model_path = "../models/airguard_xgboost_pm25.pkl"

joblib.dump(xgb_model, model_path)

print("Model saved successfully:")
print(model_path)

Model saved successfully:
../models/airguard_xgboost_pm25.pkl


In [89]:
feature_info = {
    "features": feature_columns,
    "target": target_column,
    "model": "XGBRegressor",
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

joblib.dump(
    feature_info,
    "../models/airguard_feature_config.pkl"
)

print("Feature configuration saved.")

Feature configuration saved.


In [90]:
print(os.listdir("../models"))

['airguard_feature_config.pkl', 'airguard_xgboost_pm25.pkl']
